In [ ]:
import sqlite3
from pprint import pprint

import pandas as pd

from xer_persona.scraper.config import DB_NAME, TABLE_NAME


In [30]:
TABLE_NAME

'tales'

In [28]:
conn = sqlite3.connect(DB_NAME)
df = pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)
display(df.head())

,id,titulo,origem,url,texto_completo
0,1,Androcles,Aesop,https://sites.pitt.edu/~dash/type0156.html,But shortly afterwards both Androcles and the ...
1,2,The Slave and the Lion,Aesop,https://sites.pitt.edu/~dash/type0156.html,"A slave ran away from his master, by whom he h..."
2,3,Androcles and the Lion,Joseph Jacobs,https://sites.pitt.edu/~dash/type0156.html,It happened in the old days at Rome that a sla...
3,4,The Lion and the Saint,Andrew Lang,https://sites.pitt.edu/~dash/type0156.html,"The old man with the beard is St. Jerome, who ..."
4,5,Of the Remembrance of Benefits,N/A,https://sites.pitt.edu/~dash/type0156.html,There was a knight who devoted much of his tim...


## Classificações das URLs
Com base no site de onde raspamos os contos, podemos ver que as URLs indicam diferentes tipos de contos.

In [29]:
# extrai a parte entre 'dash/' e '.html'
seg = df['url'].str.extract(r'dash/([^\.]+)\.html', expand=False)

# cria as duas novas colunas:
# - classif_other: a string inteira se NÃO começar com 'type', caso contrário NA
# - classif_atu_number: o que vem depois de 'type' se começar com 'type', caso contrário NA
df['classif_other'] = seg.where(~seg.str.startswith('type'), pd.NA)
df['classif_atu_number'] = seg.where(seg.str.startswith('type')).str.replace(r'^type', '', regex=True)
# remove leading zero from classif_atu_number when present
df['classif_atu_number'] = df['classif_atu_number'].where(
    df['classif_atu_number'].isna(),
    df['classif_atu_number'].str.replace(r'^0', '', regex=True)
)

# mostra resultado
display(df[['url', 'classif_other', 'classif_atu_number']].head(20))

,url,classif_other,classif_atu_number
0,https://sites.pitt.edu/~dash/type0156.html,<NA>,156
1,https://sites.pitt.edu/~dash/type0156.html,<NA>,156
2,https://sites.pitt.edu/~dash/type0156.html,<NA>,156
3,https://sites.pitt.edu/~dash/type0156.html,<NA>,156
4,https://sites.pitt.edu/~dash/type0156.html,<NA>,156
5,https://sites.pitt.edu/~dash/type0156.html,<NA>,156
6,https://sites.pitt.edu/~dash/grimm200.html,grimm200,NaN
7,https://sites.pitt.edu/~dash/grimm200.html,grimm200,NaN
8,https://sites.pitt.edu/~dash/fairytheft.html,fairytheft,NaN
9,https://sites.pitt.edu/~dash/fairytheft.html,fairytheft,NaN


## Adicionando as classificações ao banco de dados